# Python: NumPy, Pandas, SciPy

A refresher on the three libraries at the base of the Python data stack. **NumPy** is the n-dimensional array and the vectorized math that everything else is built on. **Pandas** is the labeled, heterogeneous, tabular layer (Series & DataFrame) for real-world data wrangling. **SciPy** is the grab-bag of *scientific* algorithms — stats, optimization, interpolation, signal processing, linear algebra, sparse matrices — that sit on top of NumPy arrays.

**Domain:** Data Analysis & Research  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**What they are.** A layered stack, each built on the one below:

- **NumPy** — the `ndarray`: a contiguous, fixed-dtype, n-dimensional block of memory plus vectorized (C-speed) elementwise math, broadcasting, slicing, reductions, and linear algebra. It is the *lingua franca* — pandas, SciPy, scikit-learn, PyTorch, and almost every numeric Python library accept or return NumPy arrays.
- **Pandas** — `Series` (1-D labeled array) and `DataFrame` (2-D table of heterogeneous, named columns) with a powerful **index**, alignment, missing-data handling, group-by, joins, reshaping, time series, and I/O for CSV/Parquet/SQL/Excel. This is where you live when the data is messy and tabular.
- **SciPy** — domain algorithms on top of NumPy: `scipy.stats` (distributions, tests), `scipy.optimize` (root-finding, curve fitting, minimization), `scipy.interpolate`, `scipy.signal`, `scipy.linalg`, `scipy.sparse`, `scipy.integrate`, `scipy.spatial`.

**The problem they solve.** Pure-Python lists and loops are slow (boxed objects, interpreter overhead) and have no notion of axes, labels, or statistical machinery. NumPy gives you **vectorization** — push the loop into compiled C so a million-element operation is one call, not a million bytecode steps. Pandas adds **labels and alignment** so you stop tracking "column 3 is age" by hand and joins/group-bys become one line. SciPy saves you from reimplementing (and getting subtly wrong) the numerical algorithms that already have battle-tested, well-documented implementations.

**When to reach for each.** NumPy for numeric arrays, math, and as the interchange format between libraries. Pandas for labeled/tabular data, ETL, exploratory analysis, time series. SciPy when you need a specific scientific algorithm (a hypothesis test, an optimizer, an interpolant, a sparse solver). **When not to:** for data bigger than RAM or heavy SQL-style aggregation, reach for **Polars / DuckDB / Dask**; for GPU/autodiff, **PyTorch / JAX**; for plain scalar code, just use Python.

## 2. Mental Model

**NumPy is one typed block of memory you do math *to all at once*; pandas is that block with sticky labels on every row and column; SciPy is the shelf of pre-written algorithms you point at the block.**

- A Python list is a box of pointers to scattered, individually-typed objects. A NumPy array is **one contiguous slab of identical-dtype values** with a *shape* (the axis lengths) and *strides* (how many bytes to step along each axis). Because the type and layout are fixed, the loop runs in C, not Python — that's the whole speed story. Slices and reshapes are usually **views** into the same slab, not copies.
- **Broadcasting** is NumPy's rule for operating on mismatched shapes without writing loops: line shapes up from the right, and any axis that is length-1 (or missing) is *virtually* stretched to match. `(3,1) + (1,4) -> (3,4)`. No data is copied; it's all index arithmetic.
- A **pandas DataFrame** is a dict of equal-length columns (each column is essentially a NumPy array) sharing a common **Index**. The index is the magic: operations **align on labels** automatically, so adding two Series matches by label, not position, and inserts `NaN` where labels don't overlap.
- **SciPy** doesn't introduce a new container — it takes NumPy arrays in and gives NumPy arrays out. Think of it as the standard library of *numerics*: you rarely hand-roll an optimizer or a t-test, you call the vetted one.

## 3. Key Concepts

- **`ndarray` (shape, dtype, strides).** The core object. `shape` = axis lengths, `dtype` = the single element type (`int64`, `float64`, `bool`, …), `strides` = byte steps per axis. `a.reshape`, `a.T`, basic slices return **views**; fancy/boolean indexing returns **copies**.
- **Vectorization.** Replace Python loops with whole-array expressions (`a * b`, `a.sum(axis=0)`, `np.where(cond, x, y)`). Orders of magnitude faster and clearer.
- **Broadcasting.** The shape-matching rules that let `array + scalar` or `(n,1) + (1,m)` work without explicit tiling. Master this and most loops disappear.
- **Axis.** `axis=0` collapses *down rows* (per-column result); `axis=1` collapses *across columns* (per-row result). The reduced axis is the one that disappears.
- **View vs copy.** Mutating a view mutates the original. `b = a[1:4]; b[0] = 99` changes `a`. Use `.copy()` when you need independence. Pandas' `SettingWithCopyWarning` is this same trap in disguise.
- **Series / DataFrame / Index.** Series = 1-D labeled array; DataFrame = dict-of-columns sharing an Index; the **Index** drives automatic **label alignment** in arithmetic and joins.
- **Vectorized pandas + `.loc`/`.iloc`.** `.loc` is **label**-based, `.iloc` is **position**-based. Prefer vectorized column expressions and `groupby`/`merge` over `.apply`/`iterrows` (which fall back to slow Python).
- **`NaN` & nullable dtypes.** Missing values are `float NaN` by default (which upcasts ints to float); pandas also has nullable `Int64`/`boolean`/`string` and `pd.NA`. Use `.isna`, `.fillna`, `.dropna`.
- **`groupby` (split-apply-combine).** Split rows by key, apply an aggregation/transform, combine back. The backbone of tabular analysis.
- **SciPy submodules.** `stats` (distributions & tests), `optimize` (`minimize`, `curve_fit`, `root`), `interpolate`, `signal`, `linalg`, `sparse`, `integrate`, `spatial` — each imported explicitly (`from scipy import stats`).

## 4. Setup

All three are pure-`pip` installs and ship as wheels for every common platform (NumPy/SciPy bundle optimized BLAS/LAPACK, so no compiler needed). Pandas pulls in NumPy automatically; SciPy needs NumPy too.

In [ ]:
# %pip install numpy pandas scipy        # uncomment if not already installed
import numpy as np
import pandas as pd
import scipy

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scipy", scipy.__version__)

## 5. Worked Examples

Four tiny, self-contained examples — all CPU-only and instant: NumPy vectorization & broadcasting, pandas wrangling with `groupby`, label alignment, and a SciPy stats + optimization combo.

### Example 1 — NumPy: vectorization, broadcasting, axes, views

The mental shift: stop writing element loops, operate on whole arrays. Note how broadcasting standardizes a matrix column-wise in one expression, and how a slice is a *view* that writes back to the original.

In [ ]:
rng = np.random.default_rng(0)          # seeded Generator -> reproducible
A = rng.integers(0, 10, size=(4, 3)).astype(float)
print("A =\n", A)

# Reductions pick an axis: axis=0 collapses rows -> one value per COLUMN.
print("\ncolumn means (axis=0):", A.mean(axis=0))
print("row sums      (axis=1):", A.sum(axis=1))

# Broadcasting: (4,3) - (3,) -> subtract per-column mean, divide per-column std.
# Standardize every column without a single loop:
A_std = (A - A.mean(axis=0)) / A.std(axis=0)
print("\nstandardized column means (~0):", A_std.mean(axis=0).round(6))
print("standardized column stds  (~1):", A_std.std(axis=0).round(6))

# Boolean masking + np.where (vectorized if/else):
print("\nelements > 5 set to 0:\n", np.where(A > 5, 0.0, A))

# View vs copy: a basic slice shares memory with A.
view = A[0]          # first row, a VIEW
view[0] = -99.0      # writes straight back into A
print("\nA[0,0] after mutating the view:", A[0, 0])

### Example 2 — Pandas: build, derive, group, aggregate

The split-apply-combine workhorse. Build a small DataFrame, add a derived column with a vectorized expression, then `groupby` to aggregate — the one-liner that replaces a pile of dict-of-list bookkeeping.

In [ ]:
df = pd.DataFrame({
    "team":   ["A", "B", "A", "B", "A", "B"],
    "player": ["w", "x", "y", "z", "p", "q"],
    "points": [10, 7, 15, 9, 12, 6],
    "minutes":[22, 30, 28, 25, 19, 31],
})

# Vectorized derived column (no loop): points scored per minute.
df["pts_per_min"] = (df["points"] / df["minutes"]).round(3)
print(df, "\n")

# Split-apply-combine: one row per team with several aggregations at once.
summary = (
    df.groupby("team")
      .agg(total_points=("points", "sum"),
           avg_ppm=("pts_per_min", "mean"),
           n_players=("player", "count"))
      .reset_index()
)
print(summary)

# Boolean filtering + .loc (label-based selection of rows AND columns):
print("\nhigh scorers:\n", df.loc[df["points"] >= 10, ["player", "points"]])

### Example 3 — Pandas: index alignment & missing data

Arithmetic between Series aligns on **labels**, not position — and inserts `NaN` where the indexes don't overlap (which silently upcasts ints to float). This is the single most surprising-then-indispensable pandas behavior.

In [ ]:
q1 = pd.Series({"alice": 100, "bob": 90, "carol": 80})
q2 = pd.Series({"bob": 95, "carol": 85, "dave": 70})   # different membership!

total = q1 + q2          # aligns on the index labels, NOT on position
print("aligned sum (note NaN where labels don't overlap):\n", total, "\n")

# Fill the gaps explicitly instead of dropping them:
total_filled = q1.add(q2, fill_value=0)
print("with fill_value=0:\n", total_filled, "\n")

# Detect / handle missing values:
print("isna mask:\n", total.isna())
print("\ndropna:\n", total.dropna())

### Example 4 — SciPy: a hypothesis test and a curve fit

SciPy on top of NumPy: run a two-sample t-test from `scipy.stats`, then fit a nonlinear model with `scipy.optimize.curve_fit` (least-squares) and recover the true parameters.

In [ ]:
from scipy import stats, optimize

# --- scipy.stats: is group B's mean different from group A's? ---
rng = np.random.default_rng(42)
a = rng.normal(loc=10.0, scale=2.0, size=50)
b = rng.normal(loc=11.5, scale=2.0, size=50)
t, p = stats.ttest_ind(a, b, equal_var=False)   # Welch's t-test (unequal variances)
print(f"Welch t-test: t = {t:.3f}, p = {p:.4g}  -> {'reject' if p < 0.05 else 'keep'} H0 at 0.05")

# --- scipy.optimize.curve_fit: recover params of y = a*exp(-b*x) + c ---
def model(x, a, b, c):
    return a * np.exp(-b * x) + c

x = np.linspace(0, 4, 60)
true = (3.0, 1.3, 0.5)
y = model(x, *true) + rng.normal(0, 0.05, size=x.size)   # noisy observations

popt, _ = optimize.curve_fit(model, x, y, p0=(1, 1, 1))
print("true params:     ", tuple(round(float(v), 3) for v in true))
print("recovered params:", tuple(round(float(v), 3) for v in popt))

## 6. Gotchas & Pitfalls

- **Views vs copies (silent mutation).** Basic slices are **views** — writing to `a[2:5]` changes the original. Fancy indexing (`a[[0,2]]`) and boolean masks return **copies**. When in doubt, `.copy()`.
- **`SettingWithCopyWarning`.** `df[df.x > 0]["y"] = 1` may write to a temporary copy and silently do nothing. Use a single `.loc`: `df.loc[df.x > 0, "y"] = 1`. (Pandas 3.0 / Copy-on-Write makes the semantics consistent, but the one-`.loc` habit is still correct.)
- **`NaN` upcasts int columns to float**, and `NaN != NaN`. Comparisons against `NaN` are always `False` — use `.isna()`, never `== np.nan`. Prefer nullable `Int64`/`boolean`/`string` dtypes when you need integers *with* missing values.
- **`.apply` / `iterrows` are slow.** They fall back to a Python loop. Reach for vectorized column math, `groupby`, `merge`, `np.where`, `np.select` first; `apply` is a last resort.
- **Integer dtype overflow & integer division.** NumPy ints are fixed-width (`int64` wraps silently on overflow, unlike Python's big ints); `int_array / int_array` returns float, `//` floors.
- **Float equality.** `0.1 + 0.2 != 0.3`. Use `np.isclose` / `np.allclose`, not `==`, for floats.
- **Broadcasting surprises.** Shapes that *happen* to be compatible can broadcast when you meant a per-element op — and a `(n,)` vs `(n,1)` mismatch yields an `(n,n)` outer result. Print `.shape` when math looks wrong.
- **Chained indexing.** `df["a"]["b"]` (two `[]`s) is both slower and ambiguous; use `df.loc[row, col]`.
- **`axis` confusion.** `axis=0` operates *down* the rows (result per column); `axis=1` *across* the columns (result per row). The named axis is the one that collapses.
- **Mutable default & in-place ops.** Many pandas methods return a new object; `inplace=True` is being de-emphasized (and never actually saves memory). Assign the result: `df = df.dropna()`.
- **`copy=False` reductions of empty axes** and **`object` dtype** columns (mixed types, or strings before the `string` dtype) kill performance and vectorization — keep columns typed.

## 7. When to Use vs Alternatives

| Tool | Best at | Weaknesses vs this stack |
|---|---|---|
| **NumPy** | Dense numeric arrays, vectorized math, the universal in-memory interchange format | No labels; single dtype per array; in-RAM only; CPU (no autodiff/GPU) |
| **Pandas** | Labeled, heterogeneous tabular data; ETL/EDA; time series; rich I/O | Memory-hungry (rule of thumb ~5–10× the file size); single-threaded for most ops; slow on very large data |
| **SciPy** | Vetted scientific algorithms (stats, optimize, interpolate, signal, sparse, linalg) | A toolbox, not a framework; for heavy ML use scikit-learn; for deep learning use PyTorch/JAX |
| **Polars** | Fast, multi-threaded, larger-than-RAM-friendly DataFrames; lazy query optimization | Younger ecosystem; different (Arrow-based) API; fewer stats niceties — see the `polars` notebook |
| **DuckDB** | SQL analytics over Parquet/CSV/Arrow, out-of-core aggregation & joins | SQL not Python-object semantics; not for row-by-row mutation — see the `duckdb` notebook |
| **Dask / Ray** | Scaling pandas/NumPy across cores or a cluster, bigger-than-memory | Distributed-systems overhead and complexity; not worth it until you actually outgrow one machine |
| **statsmodels / scikit-learn** | Formal statistical modeling / machine learning beyond SciPy's primitives | Built *on* NumPy/pandas/SciPy — complementary, not replacements |
| **PyTorch / JAX** | GPU arrays, autodiff, deep learning | Overkill for plain data wrangling; different array semantics |

**Default:** NumPy + pandas + SciPy is the right starting point for almost any analysis that fits in RAM. Stay until you have a concrete reason to leave: **out of memory or too slow → Polars / DuckDB / Dask**; **need autodiff or a GPU → PyTorch / JAX**; **need formal models → statsmodels / scikit-learn**. All of them interoperate through NumPy arrays / Arrow, so moving pieces over is incremental, not a rewrite.

## 8. Resources

- **NumPy docs & "NumPy: the absolute basics for beginners"** — the canonical entry point and reference: https://numpy.org/doc/stable/
- **Pandas docs — "10 minutes to pandas" & the User Guide** — start here, then the cookbook: https://pandas.pydata.org/docs/
- **SciPy documentation & tutorial** (per-submodule guides for stats, optimize, etc.): https://docs.scipy.org/doc/scipy/
- **"Python for Data Analysis," 3rd ed. (Wes McKinney, pandas' creator)** — free online: https://wesmckinney.com/book/
- **NumPy broadcasting rules** (the one page worth memorizing): https://numpy.org/doc/stable/user/basics.broadcasting.html
- **From Python to NumPy** (Nicolas Rougier) — a deep, free book on thinking in vectorized arrays: https://www.labri.fr/perso/nrougier/from-python-to-numpy/

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def broadcast_shape(*shapes):
    """The result shape of an elementwise operation, or ValueError if there is none."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE